# Ejercicio 9: Discusión

Aquí tocaba reflexionar sobre lo que aprendimos con DuckDB, las ventajas y limitaciones, y lo que nos llevamos del lab.

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

## 9.1 Características de DuckDB más útiles

**1. Consulta directa sobre Parquet:** DuckDB puede consultar archivos Parquet sin importarlos a tabla. Esto es lo más chido porque no tienes que esperar a que cargue todo, puedes explorar rápido.

**2. Glob patterns en read_parquet:** Puedes usar `data/raw/*/*/*.parquet` y DuckDB agarra todos los archivos automáticamente. No tienes que listar cada archivo, lo cual es un alivio.

**3. union_by_name = true:** Cuando los archivos tienen columnas nuevas (como request_source en junio 2026), DuckDB las incluye sin romper las consultas. Esto nos salvó de tener que reescribir todo.

**4. Funciones de metadatos:** `parquet_file_metadata()` te deja verificar los archivos sin leer los datos. Lo usamos para validar que las descargas estuvieran completas.

**5. Compatibilidad con pandas:** `con.sql().df()` convierte resultados directamente a DataFrame. No tienes que hacer conversiones raras.

## 9.2 Ventajas y limitaciones de consultar Parquet directamente

**Ventajas:**
- **Sin tiempo de carga:** Los datos se leen bajo demanda, ideal para explorar
- **Espacio eficiente:** No duplicas los datos en una base
- **Incremental:** Archivos nuevos entran solos sin reconfigurar
- **Flexibilidad:** Puedes cambiar las consultas sin costo

**Limitaciones:**
- **Velocidad:** Es 16-20x más lento que tabla materializada
- **Sin índices:** Cada consulta escanea todo
- **Sin transacciones:** No hay rollback ni consistencia
- **Memoria:** Consultas complejas pueden consumir mucha RAM

## 9.3 Ventajas y limitaciones de tablas materializadas en DuckDB

**Ventajas:**
- **Velocidad:** 16-20x más rápido que Parquet directo
- **Índices:** DuckDB crea índices automáticos
- **Optimización:** El motor optimiza el almacenamiento
- **Consistencia:** Los datos están en un estado conocido

**Limitaciones:**
- **Tiempo de carga:** 12-28 segundos para 30-70M filas
- **Espacio:** Duplica los datos
- **Rigidez:** Cambios en el esquema requieren recrear la tabla
- **Escritura única:** Solo un proceso puede escribir a la vez

## 9.4 Ventajas frente a cargar todo con Pandas

**1. Memoria:** DuckDB puede consultar datos que no caben en RAM, Pandas no. Con 100M filas y 20 columnas, Pandas necesitaría ~16GB RAM. DuckDB funciona con 2GB.

**2. Velocidad:** DuckDB es 10-100x más rápido que Pandas en agregaciones porque usa columnar storage y vectorización.

**3. SQL:** Las consultas son más legibles que cadenas de métodos de Pandas. `SELECT tipo, COUNT(*) FROM viajes GROUP BY tipo` es más claro que `df.groupby('tipo').size()`.

**4. Concurrencia:** DuckDB maneja múltiples consultas simultáneas, Pandas no.

**5. Tipado:** DuckDB infiere tipos correctamente desde Parquet, Pandas a veces falla con fechas y categorías.

## 9.5 Características del sistema que permiten incorporar nuevos datos con cambios mínimos

**1. Vistas sobre glob patterns:** Las vistas en `sql/00_vistas.sql` usan `data/raw/*/*/*.parquet`, así que cualquier año nuevo entra automáticamente.

**2. Separación de datos y análisis:** Los datos viven en `data/raw/` y las consultas en `sql/`. Agregar datos no requiere cambiar código.

**3. Extracción de año desde el archivo:** La vista `viajes_validos` saca el año del nombre del archivo con `split_part(filename, '/', -2)`, así que funciona con cualquier año.

**4. Script de descarga parametrizable:** El script acepta `--anios 2024 2025 2026` y baja solo lo que falta.

**5. Consultas agnósticas al año:** Las consultas agrupan por `anio` y `mes`, así que distinguen entre años sin modificación.

## 9.6 Qué parte del proceso debería automatizarse en producción

**1. Descarga de datos:** Un cron job que ejecute `download_data.py` diariamente para bajar los archivos nuevos.

**2. Verificación de integridad:** El script `--verificar` debería correr después de cada descarga y alertar si hay archivos incompletos.

**3. Materialización de tablas:** Si el tablero se consulta seguido, la tabla de DuckDB debería recrearse cada noche con los datos nuevos.

**4. Backups:** La base materializada debería respaldarse automáticamente.

**5. Monitoreo:** Alertas si el volumen de viajes cae abruptamente (posible problema de datos) o si el tiempo de consulta se dispara.

## 9.7 Decisiones de diseño importantes para mantener el proyecto reproducible

**1. Docker:** El `Dockerfile` fija las versiones de Python, DuckDB y dependencias, así cualquier persona levanta el mismo ambiente.

**2. Datos fuera de Git:** Los archivos Parquet no se versionan, solo los scripts que los descargan. Esto mantiene el repo ligero.

**3. SQL separado de Python:** Las consultas viven en `sql/` y se ejecutan por nombre, así la consulta documentada y la ejecutada son la misma.

**4. Rutas relativas:** `scripts/consultas.py` resuelve rutas desde la raíz del laboratorio, así los cuadernos funcionan desde cualquier directorio.

**5. Script de descarga verificable:** El script compara lo descargado con lo publicado por el servidor.

**6. Notebooks numerados:** El orden de ejecución es explícito (01, 02, 03...), así cualquiera puede reproducir el análisis.

## 9.8 Lo aprendido sobre manejo de datos que no sería evidente con conjuntos pequeños

**1. La calidad de datos importa más a escala:** Con 100M filas, problemas como horas nulas, montos negativos o fechas fuera de mes afectan miles de registros. En conjuntos pequeños estos problemas pasan desapercibidos.

**2. El costo de materialización se amortiza:** Crear una tabla tarda 12-28 segundos, pero si se consulta 100 veces, el ahorro de tiempo es enorme. Con conjuntos pequeños este tradeoff no es evidente.

**3. Los patrones temporales requieren múltiples años:** La estacionalidad y tendencias solo son visibles con datos de 2024, 2025 y 2026. Un solo año no muestra el panorama completo.

**4. La segmentación del mercado es estable:** La brecha entre amarillos y verdes se mantiene constante en los tres años, lo que sugiere que son mercados distintos que no convergen.

**5. El diseño incremental es esencial:** Agregar 2024 y 2025 sin tocar consultas solo fue posible porque el sistema fue diseñado para crecer. En conjuntos pequeños es tentador hacer soluciones ad hoc que no escalan.

In [1]:
import sys
import os
from pathlib import Path
sys.path.insert(0, str(Path('/workspace/scripts')))

import duckdb
import pandas as pd
import consultas

os.chdir(Path('/workspace'))
print(f"DuckDB version: {duckdb.__version__}")

con = consultas.conectar()

archivos = con.sql("""
    select count(*) as total
    from parquet_file_metadata('data/raw/*/*/*.parquet')
""").df()

print(f"Total de archivos: {archivos['total'].values[0]}")

DuckDB version: 1.5.5
Total de archivos: 64


In [2]:
resumen_anio = con.sql("""
    select
        split_part(file_name, '/', -2) as anio,
        count(*) as archivos,
        sum(num_rows) as registros
    from parquet_file_metadata('data/raw/*/*/*.parquet')
    group by split_part(file_name, '/', -2)
    order by anio
""").df()

print("\n=== Resumen por año ===")
display(resumen_anio)


=== Resumen por año ===


,anio,archivos,registros
0,2024,24,41829938.0
1,2025,24,49313977.0
2,2026,16,30040469.0


In [3]:
velocidad_anual = con.sql("""
    select
        tipo,
        anio,
        round(avg(trip_distance / (duracion_min / 60)), 1) as velocidad_promedio_mph
    from viajes_validos
    where medible and duracion_min > 0
    group by tipo, anio
    order by tipo, anio
""").df()

print("\n=== Comparación rápida de velocidades ===")
display(velocidad_anual)


=== Comparación rápida de velocidades ===


,tipo,anio,velocidad_promedio_mph
0,green,2024,15.8
1,green,2025,20.2
2,green,2026,15.6
3,yellow,2024,11.3
4,yellow,2025,11.3
5,yellow,2026,11.1
